# 08 - MLflow Tracking Demo

**Objectif** : Démontrer le tracking MLflow pour les modèles DeepPilot.

Ce notebook montre :
1. Configuration MLflow
2. Tracking des expériences HMM et Random Forest
3. Comparaison de runs
4. Model Registry

**Compétence validée** : C11 (MLOps)

In [17]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow

# Modules DeepPilot - MLOps
from mlops import (
    start_run, log_params, log_metrics, log_model,
    log_figure, get_best_run, compare_runs,
    MLFLOW_CONFIG, MODEL_NAMES, PERFORMANCE_THRESHOLDS
)
from mlops.registry import register_model, get_model_versions, promote_model

# Modules DeepPilot - ML
from ml.features import prepare_regime_features, prepare_prediction_features, create_target
from ml.models.regime_hmm import RegimeHMM

# Data extractors
from data.extractors.extract_yfinance import download_etf_prices, ETF_TICKERS

print(f"MLflow version: {mlflow.__version__}")
print(f"Tracking URI: {MLFLOW_CONFIG.tracking_uri}")

MLflow version: 3.12.0
Tracking URI: file:///c:/Users/User/OneDrive/Desktop/deepilot/analysis/mlruns


## 1. Chargement des données

In [18]:
# Charger les prix ETF
from datetime import datetime

prices = download_etf_prices(ETF_TICKERS, start_date="2010-01-01", end_date=datetime.now().strftime("%Y-%m-%d"))
print(f"Prix ETF: {prices.shape}")
print(f"Période: {prices.index.min()} à {prices.index.max()}")
prices.tail()

2026-05-15 21:28:56,675 - INFO - Téléchargement de 8 tickers: ['SPY', 'EFA', 'EEM', 'TLT', 'HYG', 'GLD', 'VNQ', 'SH']
2026-05-15 21:28:56,676 - INFO - Période: 2010-01-01 → 2026-05-15
[*********************100%***********************]  8 of 8 completed
2026-05-15 21:28:56,959 - INFO - Tickers téléchargés avec succès: ['SPY', 'EFA', 'EEM', 'TLT', 'HYG', 'GLD', 'VNQ', 'SH']
2026-05-15 21:28:56,960 - INFO - Nombre de lignes: 4116


Prix ETF: (4116, 8)
Période: 2010-01-04 00:00:00 à 2026-05-14 00:00:00


Ticker,EEM,EFA,GLD,HYG,SH,SPY,TLT,VNQ
Date,,,,,,,,
2026-05-08,67.940002,103.959999,433.769989,80.139999,33.610001,737.619995,86.080002,96.620003
2026-05-11,67.889999,103.739998,434.649994,79.980003,33.529999,739.299988,85.559998,96.690002
2026-05-12,65.820000,103.139999,432.929993,79.870003,33.590000,738.179993,84.989998,96.690002
2026-05-13,67.209999,103.830002,430.500000,79.910004,33.410000,742.309998,84.800003,95.889999
2026-05-14,67.379997,103.459999,427.209991,79.849998,33.139999,748.169983,84.919998,95.320000


In [19]:
# Charger les données macro depuis Supabase
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

DATABASE_URL = os.getenv("SUPABASE_DB_URL")
engine = create_engine(DATABASE_URL)

query_macro = """
SELECT date, vix, credit_spread_hy, yield_curve_10y2y
FROM macro_indicator
ORDER BY date
"""

macro_df = pd.read_sql(query_macro, engine, parse_dates=['date'])
macro_df = macro_df.set_index('date')

# Remplir les valeurs manquantes
if macro_df['credit_spread_hy'].isna().any():
    macro_df['credit_spread_hy'] = macro_df['credit_spread_hy'].ffill().bfill().fillna(4.5)
    print("[INFO] credit_spread_hy: valeurs manquantes remplies")

print(f"Données macro: {macro_df.shape}")
macro_df.head()

[INFO] credit_spread_hy: valeurs manquantes remplies
Données macro: (4326, 3)


,vix,credit_spread_hy,yield_curve_10y2y
date,,,
2010-01-01,NaN,2.54,NaN
2010-01-04,20.04,2.54,2.76
2010-01-05,19.35,2.53,2.76
2010-01-06,19.16,2.49,2.84
2010-01-07,19.06,2.48,2.82


In [20]:
# Combiner prix et macro pour les features
combined_df = prices.join(macro_df, how='inner')
print(f"Données combinées: {combined_df.shape}")

# Préparer les features pour le régime
regime_features = prepare_regime_features(combined_df)
print(f"Features régime: {regime_features.shape}")
regime_features.head()

Données combinées: (4116, 11)
Features régime: (4116, 5)


,vix_zscore,credit_spread_zscore,yield_curve_10y2y,spy_return_20d,spy_volatility_20d
2010-01-04,0.590802,0.0,2.76,-0.02603,0.113785
2010-01-05,0.590802,0.0,2.76,-0.02603,0.113785
2010-01-06,0.590802,0.0,2.84,-0.02603,0.113785
2010-01-07,0.590802,0.0,2.82,-0.02603,0.113785
2010-01-08,0.590802,0.0,2.87,-0.02603,0.113785


## 2. Tracking HMM - Détection de régimes

On compare plusieurs configurations de HMM avec MLflow.

In [21]:
# Préparer les données pour HMM
X_hmm = regime_features.dropna()
print(f"Données HMM: {X_hmm.shape}")
X_hmm.head()

Données HMM: (4116, 5)


,vix_zscore,credit_spread_zscore,yield_curve_10y2y,spy_return_20d,spy_volatility_20d
2010-01-04,0.590802,0.0,2.76,-0.02603,0.113785
2010-01-05,0.590802,0.0,2.76,-0.02603,0.113785
2010-01-06,0.590802,0.0,2.84,-0.02603,0.113785
2010-01-07,0.590802,0.0,2.82,-0.02603,0.113785
2010-01-08,0.590802,0.0,2.87,-0.02603,0.113785


In [22]:
# Grid search avec tracking MLflow
hmm_configs = [
    {"n_regimes": 3, "n_iter": 50, "covariance_type": "full"},
    {"n_regimes": 4, "n_iter": 50, "covariance_type": "full"},
    {"n_regimes": 4, "n_iter": 100, "covariance_type": "full"},
    {"n_regimes": 5, "n_iter": 100, "covariance_type": "diag"},
]

print("Tracking HMM experiments...")
for i, config in enumerate(hmm_configs):
    with start_run(experiment="regime_detection", run_name=f"hmm_config_{i+1}") as run:
        # Log params
        log_params(config)
        log_params({"features": list(X_hmm.columns), "n_samples": len(X_hmm)})
        
        # Train HMM
        hmm = RegimeHMM(
            n_regimes=config["n_regimes"],
            n_iter=config["n_iter"],
            covariance_type=config["covariance_type"]
        )
        hmm.fit(X_hmm)
        
        # Predict and evaluate
        regimes = hmm.predict(X_hmm)
        
        # Calculer métriques
        from sklearn.metrics import silhouette_score
        silhouette = silhouette_score(X_hmm, regimes)
        
        # Stabilité : % de jours sans changement de régime
        regime_changes = np.sum(np.diff(regimes) != 0)
        stability = 1 - (regime_changes / len(regimes))
        
        # Log metrics
        log_metrics({
            "silhouette": silhouette,
            "stability": stability,
            "log_likelihood": hmm.model.score(X_hmm),
            "n_regime_changes": regime_changes,
        })
        
        # Log model
        log_model(hmm, "hmm_model")
        
        print(f"  Config {i+1}: silhouette={silhouette:.3f}, stability={stability:.3f}")

2026-05-15 21:28:57,579 - INFO - MLflow tracking URI: file:///c:/Users/User/OneDrive/Desktop/deepilot/analysis/mlruns
2026-05-15 21:28:57,580 - INFO - MLflow tracking URI: file:///c:/Users/User/OneDrive/Desktop/deepilot/analysis/mlruns
2026-05-15 21:28:57,589 - INFO - Expérience existante: deeppilot-regime (ID: 508446076668040779)
2026-05-15 21:28:57,664 - INFO - Run démarré: c747bce1e5aa4fb6a465248c744c11f4


Tracking HMM experiments...


2026/05/15 21:28:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/15 21:28:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026-05-15 21:29:02,321 - INFO - Modèle loggé: hmm_model (type: sklearn)
2026-05-15 21:29:02,326 - INFO - Run terminé: c747bce1e5aa4fb6a465248c744c11f4
2026-05-15 21:29:02,327 - INFO - MLflow tracking URI: file:///c:/Users/User/OneDrive/Desktop/deepilot/analysis/mlruns
2026-05-15 21:29:02,328 - INFO - MLflow tracking URI: file:///c:/Users/User/OneDrive/Desktop/deepilot/analysis/mlruns
2026-05-15 21:29:02,333 - INFO - Expérience existante: deeppilot-regime (ID: 508446076668040779)
2026-05

  Config 1: silhouette=0.121, stability=0.559


2026/05/15 21:29:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/15 21:29:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026-05-15 21:29:06,773 - INFO - Modèle loggé: hmm_model (type: sklearn)
2026-05-15 21:29:06,778 - INFO - Run terminé: 3feec87a09304b47a1f8017525ff98ce
2026-05-15 21:29:06,779 - INFO - MLflow tracking URI: file:///c:/Users/User/OneDrive/Desktop/deepilot/analysis/mlruns
2026-05-15 21:29:06,779 - INFO - MLflow tracking URI: file:///c:/Users/User/OneDrive/Desktop/deepilot/analysis/mlruns
2026-05-15 21:29:06,787 - INFO - Expérience existante: deeppilot-regime (ID: 508446076668040779)
2026-05

  Config 2: silhouette=0.034, stability=0.967


2026/05/15 21:29:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/15 21:29:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026-05-15 21:29:11,218 - INFO - Modèle loggé: hmm_model (type: sklearn)
2026-05-15 21:29:11,223 - INFO - Run terminé: 4efd86a19ad545529e5b71378ee2e0f8
2026-05-15 21:29:11,224 - INFO - MLflow tracking URI: file:///c:/Users/User/OneDrive/Desktop/deepilot/analysis/mlruns
2026-05-15 21:29:11,224 - INFO - MLflow tracking URI: file:///c:/Users/User/OneDrive/Desktop/deepilot/analysis/mlruns
2026-05-15 21:29:11,232 - INFO - Expérience existante: deeppilot-regime (ID: 508446076668040779)
2026-05

  Config 3: silhouette=0.034, stability=0.967


2026/05/15 21:29:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/15 21:29:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026-05-15 21:29:15,816 - INFO - Modèle loggé: hmm_model (type: sklearn)
2026-05-15 21:29:15,821 - INFO - Run terminé: c9a180615d064c6897135d274603b525


  Config 4: silhouette=-0.011, stability=0.952


In [23]:
# Trouver le meilleur run
best_hmm = get_best_run("regime_detection", metric="silhouette", mode="max")
if best_hmm:
    print("Meilleur run HMM:")
    print(f"  Run ID: {best_hmm['run_id']}")
    print(f"  Métriques: {best_hmm['metrics']}")
    print(f"  Params: {best_hmm['params']}")

2026-05-15 21:29:15,832 - INFO - MLflow tracking URI: file:///c:/Users/User/OneDrive/Desktop/deepilot/analysis/mlruns
2026-05-15 21:29:15,834 - INFO - MLflow tracking URI: file:///c:/Users/User/OneDrive/Desktop/deepilot/analysis/mlruns
2026-05-15 21:29:15,842 - INFO - Expérience existante: deeppilot-regime (ID: 508446076668040779)


Meilleur run HMM:
  Run ID: 17229dd4026b413ba37301f01daa861e
  Métriques: {'log_likelihood': -29027.80820718375, 'n_regime_changes': 1816.0, 'silhouette': 0.12072494593548905, 'stability': 0.5587949465500486}
  Params: {'covariance_type': 'full', 'features': '["vix_zscore", "credit_spread_zscore", "yield_curve_10y2y", "spy_return_20d", "spy_volatility_20d"]', 'n_iter': '50', 'n_regimes': '3', 'n_samples': '4116'}


## 3. Tracking Random Forest - Prédiction

On compare plusieurs configurations de Random Forest.

In [24]:
# Préparer les données pour RF
# Features de prédiction basées sur SPY
pred_features = prepare_prediction_features(combined_df, "SPY")

# Target : rendement positif à 21 jours
target = create_target(combined_df, "SPY", horizon=21)

# Aligner features et target
common_idx = pred_features.index.intersection(target.dropna().index)
X_rf = pred_features.loc[common_idx]
y_rf = target.loc[common_idx]

print(f"Données RF: X={X_rf.shape}, y={len(y_rf)}")
print(f"Distribution target: {y_rf.value_counts().to_dict()}")

Données RF: X=(3917, 10), y=3917
Distribution target: {1: 2691, 0: 1226}


In [25]:
# Grid search RF avec tracking
rf_configs = [
    {"n_estimators": 50, "max_depth": 5, "min_samples_split": 20},
    {"n_estimators": 100, "max_depth": 5, "min_samples_split": 20},
    {"n_estimators": 100, "max_depth": 8, "min_samples_split": 10},
    {"n_estimators": 200, "max_depth": 10, "min_samples_split": 5},
]

# Train/test split temporel
split_date = "2020-01-01"
X_train = X_rf[X_rf.index < split_date]
X_test = X_rf[X_rf.index >= split_date]
y_train = y_rf[y_rf.index < split_date]
y_test = y_rf[y_rf.index >= split_date]

print(f"Train: {len(X_train)}, Test: {len(X_test)}")

print("\nTracking RF experiments...")
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

for i, config in enumerate(rf_configs):
    with start_run(experiment="prediction", run_name=f"rf_config_{i+1}") as run:
        # Log params
        log_params(config)
        log_params({"n_features": X_rf.shape[1], "train_size": len(X_train), "test_size": len(X_test)})
        
        # Train model
        rf = RandomForestClassifier(**config, random_state=42, n_jobs=-1)
        rf.fit(X_train, y_train)
        
        # Evaluate
        y_pred = rf.predict(X_test)
        y_proba = rf.predict_proba(X_test)[:, 1]
        
        accuracy = accuracy_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_proba)
        f1 = f1_score(y_test, y_pred)
        
        # Log metrics
        log_metrics({
            "accuracy": accuracy,
            "auc": auc,
            "f1": f1,
        })
        
        # Log feature importance
        importance = pd.Series(rf.feature_importances_, index=X_rf.columns).sort_values(ascending=False)
        
        # Plot feature importance
        fig, ax = plt.subplots(figsize=(10, 6))
        importance.plot(kind='barh', ax=ax)
        ax.set_title(f"Feature Importance (Config {i+1})")
        ax.set_xlabel("Importance")
        log_figure(fig, "feature_importance.png")
        plt.close()
        
        # Log model
        log_model(rf, "rf_model")
        
        print(f"  Config {i+1}: accuracy={accuracy:.3f}, AUC={auc:.3f}, F1={f1:.3f}")

2026-05-15 21:29:16,100 - INFO - MLflow tracking URI: file:///c:/Users/User/OneDrive/Desktop/deepilot/analysis/mlruns
2026-05-15 21:29:16,101 - INFO - MLflow tracking URI: file:///c:/Users/User/OneDrive/Desktop/deepilot/analysis/mlruns
2026-05-15 21:29:16,109 - INFO - Expérience existante: deeppilot (ID: 776691368110299590)
2026-05-15 21:29:16,177 - INFO - Run démarré: dee797453caa481aa8faccd93ea5826d


Train: 2317, Test: 1600

Tracking RF experiments...


2026-05-15 21:29:16,535 - INFO - Artifact loggé: C:\Users\User\AppData\Local\Temp\tmpufwf5xs0\feature_importance.png
2026/05/15 21:29:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/15 21:29:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026-05-15 21:29:20,279 - INFO - Modèle loggé: rf_model (type: sklearn)
2026-05-15 21:29:20,284 - INFO - Run terminé: dee797453caa481aa8faccd93ea5826d
2026-05-15 21:29:20,285 - INFO - MLflow tracking URI: file:///c:/Users/User/OneDrive/Desktop/deepilot/analysis/mlruns
2026-05-15 21:29:20,285 - INFO - MLflow tracking URI: file:///c:/Users/User/OneDrive/Desktop/deepilot/anal

  Config 1: accuracy=0.608, AUC=0.495, F1=0.736


2026-05-15 21:29:20,765 - INFO - Artifact loggé: C:\Users\User\AppData\Local\Temp\tmpqx0d0dgx\feature_importance.png
2026/05/15 21:29:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/15 21:29:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026-05-15 21:29:24,533 - INFO - Modèle loggé: rf_model (type: sklearn)
2026-05-15 21:29:24,537 - INFO - Run terminé: 1f08de7c86ee4bd3a8fa2c01a2a4eb71
2026-05-15 21:29:24,538 - INFO - MLflow tracking URI: file:///c:/Users/User/OneDrive/Desktop/deepilot/analysis/mlruns
2026-05-15 21:29:24,538 - INFO - MLflow tracking URI: file:///c:/Users/User/OneDrive/Desktop/deepilot/anal

  Config 2: accuracy=0.613, AUC=0.498, F1=0.739


2026-05-15 21:29:25,006 - INFO - Artifact loggé: C:\Users\User\AppData\Local\Temp\tmp7fsbxnvw\feature_importance.png
2026/05/15 21:29:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/15 21:29:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026-05-15 21:29:28,608 - INFO - Modèle loggé: rf_model (type: sklearn)
2026-05-15 21:29:28,614 - INFO - Run terminé: 99032afdd5c8454aa0f7dd1f514ef71a
2026-05-15 21:29:28,615 - INFO - MLflow tracking URI: file:///c:/Users/User/OneDrive/Desktop/deepilot/analysis/mlruns
2026-05-15 21:29:28,616 - INFO - MLflow tracking URI: file:///c:/Users/User/OneDrive/Desktop/deepilot/anal

  Config 3: accuracy=0.604, AUC=0.514, F1=0.727


2026-05-15 21:29:29,280 - INFO - Artifact loggé: C:\Users\User\AppData\Local\Temp\tmpvo6boxj3\feature_importance.png
2026/05/15 21:29:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/15 21:29:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026-05-15 21:29:32,881 - INFO - Modèle loggé: rf_model (type: sklearn)
2026-05-15 21:29:32,886 - INFO - Run terminé: c8a4720346c94e1e9679ddf365e10824


  Config 4: accuracy=0.600, AUC=0.514, F1=0.721


In [26]:
# Comparer les runs RF
print("Comparaison des runs RF:")
comparison = compare_runs("prediction", metric="auc")
if comparison:
    for run in comparison:
        print(f"  {run['run_name']}: AUC={run['metrics'].get('auc', 'N/A'):.3f}")

2026-05-15 21:29:32,895 - INFO - MLflow tracking URI: file:///c:/Users/User/OneDrive/Desktop/deepilot/analysis/mlruns
2026-05-15 21:29:32,897 - INFO - MLflow tracking URI: file:///c:/Users/User/OneDrive/Desktop/deepilot/analysis/mlruns
2026-05-15 21:29:32,906 - INFO - Expérience existante: deeppilot (ID: 776691368110299590)


Comparaison des runs RF:
  rf_config_4: AUC=0.514
  rf_config_4: AUC=0.514
  rf_config_4: AUC=0.514
  rf_config_3: AUC=0.514
  rf_config_3: AUC=0.514


In [27]:
# Trouver le meilleur run RF
best_rf = get_best_run("prediction", metric="auc", mode="max")
if best_rf:
    print("Meilleur run RF:")
    print(f"  Run ID: {best_rf['run_id']}")
    print(f"  Métriques: {best_rf['metrics']}")
    print(f"  Params: {best_rf['params']}")

2026-05-15 21:29:33,098 - INFO - MLflow tracking URI: file:///c:/Users/User/OneDrive/Desktop/deepilot/analysis/mlruns
2026-05-15 21:29:33,100 - INFO - MLflow tracking URI: file:///c:/Users/User/OneDrive/Desktop/deepilot/analysis/mlruns
2026-05-15 21:29:33,107 - INFO - Expérience existante: deeppilot (ID: 776691368110299590)


Meilleur run RF:
  Run ID: c8a4720346c94e1e9679ddf365e10824
  Métriques: {'accuracy': 0.6, 'auc': 0.5144176680926686, 'f1': 0.7205240174672489}
  Params: {'max_depth': '10', 'min_samples_split': '5', 'n_estimators': '200', 'n_features': '10', 'test_size': '1600', 'train_size': '2317'}


## 4. Model Registry

On enregistre les meilleurs modèles dans le registry MLflow.

In [28]:
# Enregistrer le meilleur modèle HMM
if best_hmm:
    try:
        model_version = register_model(
            run_id=best_hmm['run_id'],
            artifact_path="hmm_model",
            model_name=MODEL_NAMES['regime']
        )
        print(f"Modèle HMM enregistré: {MODEL_NAMES['regime']} version {model_version}")
    except Exception as e:
        print(f"Note: {e}")

2026-05-15 21:29:33,199 - INFO - MLflow tracking URI: file:///c:/Users/User/OneDrive/Desktop/deepilot/analysis/mlruns
Registered model 'deeppilot-regime-hmm' already exists. Creating a new version of this model...
2026/05/15 21:29:33 WARNING mlflow.tracking._model_registry.fluent: Run with id 17229dd4026b413ba37301f01daa861e has no artifacts at artifact path 'hmm_model', registering model based on models:/m-e68db5068be946b9bef364a5a69ba45a instead
Created version '3' of model 'deeppilot-regime-hmm'.
2026-05-15 21:29:33,376 - INFO - Modèle enregistré: deeppilot-regime-hmm v3


Modèle HMM enregistré: deeppilot-regime-hmm version <ModelVersion: aliases=[], creation_timestamp=1778873373300, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1778873373300, metrics=[<Metric: dataset_digest=None, dataset_name=None, key='log_likelihood', model_id='m-e68db5068be946b9bef364a5a69ba45a', run_id='17229dd4026b413ba37301f01daa861e', step=0, timestamp=1778871346569, value=-29027.80820718375>,
 <Metric: dataset_digest=None, dataset_name=None, key='n_regime_changes', model_id='m-e68db5068be946b9bef364a5a69ba45a', run_id='17229dd4026b413ba37301f01daa861e', step=0, timestamp=1778871346569, value=1816.0>,
 <Metric: dataset_digest=None, dataset_name=None, key='silhouette', model_id='m-e68db5068be946b9bef364a5a69ba45a', run_id='17229dd4026b413ba37301f01daa861e', step=0, timestamp=1778871346569, value=0.12072494593548905>,
 <Metric: dataset_digest=None, dataset_name=None, key='stability', model_id='m-e68db5068be946b9bef364a5a69ba45a', run_id=

In [29]:
# Enregistrer le meilleur modèle RF
if best_rf:
    try:
        model_version = register_model(
            run_id=best_rf['run_id'],
            artifact_path="rf_model",
            model_name=MODEL_NAMES['prediction']
        )
        print(f"Modèle RF enregistré: {MODEL_NAMES['prediction']} version {model_version}")
    except Exception as e:
        print(f"Note: {e}")

2026-05-15 21:29:33,386 - INFO - MLflow tracking URI: file:///c:/Users/User/OneDrive/Desktop/deepilot/analysis/mlruns
Registered model 'deeppilot-prediction-rf' already exists. Creating a new version of this model...
2026/05/15 21:29:33 WARNING mlflow.tracking._model_registry.fluent: Run with id c8a4720346c94e1e9679ddf365e10824 has no artifacts at artifact path 'rf_model', registering model based on models:/m-2764da01f399490fa37df2d3eb6be378 instead
Created version '3' of model 'deeppilot-prediction-rf'.
2026-05-15 21:29:33,569 - INFO - Modèle enregistré: deeppilot-prediction-rf v3


Modèle RF enregistré: deeppilot-prediction-rf version <ModelVersion: aliases=[], creation_timestamp=1778873373493, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1778873373493, metrics=[<Metric: dataset_digest=None, dataset_name=None, key='accuracy', model_id='m-2764da01f399490fa37df2d3eb6be378', run_id='c8a4720346c94e1e9679ddf365e10824', step=0, timestamp=1778873369079, value=0.6>,
 <Metric: dataset_digest=None, dataset_name=None, key='auc', model_id='m-2764da01f399490fa37df2d3eb6be378', run_id='c8a4720346c94e1e9679ddf365e10824', step=0, timestamp=1778873369079, value=0.5144176680926686>,
 <Metric: dataset_digest=None, dataset_name=None, key='f1', model_id='m-2764da01f399490fa37df2d3eb6be378', run_id='c8a4720346c94e1e9679ddf365e10824', step=0, timestamp=1778873369079, value=0.7205240174672489>], model_id='m-2764da01f399490fa37df2d3eb6be378', name='deeppilot-prediction-rf', params={'max_depth': '10',
 'min_samples_split': '5',
 'n_estimators':

In [30]:
# Lister les versions
print("\nVersions du modèle HMM:")
try:
    versions = get_model_versions(MODEL_NAMES['regime'])
    for v in versions:
        print(f"  Version {v['version']}: stage={v['stage']}, created={v['creation_timestamp']}")
except Exception as e:
    print(f"  Aucune version enregistrée: {e}")

2026-05-15 21:29:33,579 - INFO - MLflow tracking URI: file:///c:/Users/User/OneDrive/Desktop/deepilot/analysis/mlruns



Versions du modèle HMM:
  Aucune version enregistrée: 'ModelVersion' object is not subscriptable


## 5. MLflow UI

Pour visualiser les expériences dans l'interface MLflow :

```bash
cd deeppilot
python -m mlflow ui --port 5000
```

Puis ouvrir http://localhost:5000 dans un navigateur.

> **Note** : Si `mlflow` n'est pas reconnu comme commande, utiliser `python -m mlflow` fonctionne sans avoir besoin d'activer un virtual environment.

In [31]:
print("Pour lancer MLflow UI:")
print("  cd deeppilot")
print("  python -m mlflow ui --port 5000")
print("\nPuis ouvrir: http://localhost:5000")

Pour lancer MLflow UI:
  cd deeppilot
  python -m mlflow ui --port 5000

Puis ouvrir: http://localhost:5000


## Résumé

Ce notebook a démontré :

1. **Tracking des expériences** : logging des hyperparamètres, métriques, et modèles
2. **Comparaison de runs** : identification du meilleur modèle par métrique
3. **Model Registry** : enregistrement et versioning des modèles

**Compétence C11 validée** : MLOps avec MLflow